# API Sports to Matches_Doubled.csv

## Dependencies

In [34]:
from packages.helpers.helpers import joel_boto
import io
import os
import requests
import pandas as pd
from datetime import datetime

API_KEY = os.getenv("api_sports_api")
if not API_KEY:
    raise RuntimeError("Please set env var api_sports_api to your API-SPORTS API key")

BASE_URL = "https://v1.american-football.api-sports.io"

HEADERS = {
    "x-apisports-key": API_KEY
}

## Games

In [51]:
def get_nfl_games_df() -> list:

    season = datetime.now().year

    url = f"{BASE_URL}/games"
    
    params = {
        "league": 1,      # ✅ NFL = league 1
        "season": season
    }

    r = requests.get(url, headers=HEADERS, params=params, timeout=20)
    r.raise_for_status()

    games = r.json()["response"]

    df = pd.json_normalize(games)

    return df


In [95]:
games_df = get_nfl_games_df()
games_df.columns

Index(['game.id', 'game.stage', 'game.week', 'game.date.timezone',
       'game.date.date', 'game.date.time', 'game.date.timestamp',
       'game.venue.name', 'game.venue.city', 'game.status.short',
       'game.status.long', 'game.status.timer', 'league.id', 'league.name',
       'league.season', 'league.logo', 'league.country.name',
       'league.country.code', 'league.country.flag', 'teams.home.id',
       'teams.home.name', 'teams.home.logo', 'teams.away.id',
       'teams.away.name', 'teams.away.logo', 'scores.home.quarter_1',
       'scores.home.quarter_2', 'scores.home.quarter_3',
       'scores.home.quarter_4', 'scores.home.overtime', 'scores.home.total',
       'scores.away.quarter_1', 'scores.away.quarter_2',
       'scores.away.quarter_3', 'scores.away.quarter_4',
       'scores.away.overtime', 'scores.away.total'],
      dtype='object')

## Stats

In [78]:
def get_game_team_stats(game_id: int):
    url = f"{BASE_URL}/games/statistics/teams"
    params = {
        "id": game_id
    }

    r = requests.get(url, headers=HEADERS, params=params, timeout=20)
    r.raise_for_status()
    data = r.json()
    stats = data.get("response", [])
    df = pd.json_normalize(stats)
    return df

In [99]:
game_id = int(games_df["game.id"].iloc[189])
game_id 

17454

In [100]:
stats = get_game_team_stats(game_id)
stats

,team.id,team.name,team.logo,statistics.first_downs.total,statistics.first_downs.passing,statistics.first_downs.rushing,statistics.first_downs.from_penalties,statistics.first_downs.third_down_efficiency,statistics.first_downs.fourth_down_efficiency,statistics.plays.total,...,statistics.turnovers.total,statistics.turnovers.lost_fumbles,statistics.turnovers.interceptions,statistics.posession.total,statistics.interceptions.total,statistics.fumbles_recovered.total,statistics.sacks.total,statistics.safeties.total,statistics.int_touchdowns.total,statistics.points_against.total
0,13,New York Jets,https://media.api-sports.io/american-football/...,12,1,8,3,3-11,0-0,47,...,1,0,1,25:56,0,0,6,0,2,20
1,9,Cleveland Browns,https://media.api-sports.io/american-football/...,23,8,10,5,6-16,0-1,70,...,0,0,0,34:04,1,0,3,0,0,27


## Events

In [103]:
def get_game_events(game_id: int):
    url = f"{BASE_URL}/games/events"
    params = {
        "id": game_id
    }

    r = requests.get(url, headers=HEADERS, params=params, timeout=20)
    r.raise_for_status()
    data = r.json()
    stats = data.get("response", [])
    df = pd.json_normalize(stats)
    return df

In [104]:
events = get_game_events(game_id)
events

,quarter,minute,type,comment,team.id,team.name,team.logo,player.id,player.name,player.image,score.home,score.away
0,First,None,TD,David Njoku 9 Yd pass from Dillon Gabriel (And...,9,Cleveland Browns,https://media.api-sports.io/american-football/...,630,David Njoku,https://media.api-sports.io/american-football/...,0,7
1,First,None,TD,Kene Nwangwu 99 Yd Kickoff Return (Nick Folk K...,13,New York Jets,https://media.api-sports.io/american-football/...,2314,Kene Nwangwu,https://media.api-sports.io/american-football/...,7,7
2,First,None,TD,Isaiah Williams 74 Yd Punt Return (Nick Folk K...,13,New York Jets,https://media.api-sports.io/american-football/...,7309,Isaiah Williams,https://media.api-sports.io/american-football/...,14,7
3,Second,None,TD,Jerry Jeudy 22 Yd pass from Dillon Gabriel (An...,9,Cleveland Browns,https://media.api-sports.io/american-football/...,2009,Jerry Jeudy,https://media.api-sports.io/american-football/...,14,14
4,Second,None,FG,Nick Folk 26 Yd Field Goal,13,New York Jets,https://media.api-sports.io/american-football/...,211,Nick Folk,https://media.api-sports.io/american-football/...,17,14
5,Second,None,FG,Andre Szmyt 45 Yd Field Goal,9,Cleveland Browns,https://media.api-sports.io/american-football/...,24720,Andre Szmyt,https://media.api-sports.io/american-football/...,17,17
6,Fourth,None,TD,Breece Hall 42 Yd pass from Justin Fields (Nic...,13,New York Jets,https://media.api-sports.io/american-football/...,907,Breece Hall,https://media.api-sports.io/american-football/...,24,17
7,Fourth,None,FG,Nick Folk 37 Yd Field Goal,13,New York Jets,https://media.api-sports.io/american-football/...,211,Nick Folk,https://media.api-sports.io/american-football/...,27,17
8,Fourth,None,FG,Andre Szmyt 29 Yd Field Goal,9,Cleveland Browns,https://media.api-sports.io/american-football/...,24720,Andre Szmyt,https://media.api-sports.io/american-football/...,27,20


## Players

In [106]:
def get_game_player_stats(game_id: int):
    url = f"{BASE_URL}/games/statistics/players"
    params = {
        "id": game_id
    }

    r = requests.get(url, headers=HEADERS, params=params, timeout=20)
    r.raise_for_status()
    data = r.json()
    stats = data.get("response", [])
    df = pd.json_normalize(stats)
    return df

In [125]:
player_stats = get_game_player_stats(game_id)
pd.json_normalize(player_stats[:1]["groups"][0][7]['players'][0]['statistics'])

,name,value
0,total,5
1,yards,249
2,average,49.8
3,touchbacks,0
4,in20,3
5,lg,55


Index(['game.id', 'game.stage', 'game.week', 'game.date.timezone',
       'game.date.date', 'game.date.time', 'game.date.timestamp',
       'game.venue.name', 'game.venue.city', 'game.status.short',
       'game.status.long', 'game.status.timer', 'league.id', 'league.name',
       'league.season', 'league.logo', 'league.country.name',
       'league.country.code', 'league.country.flag', 'teams.home.id',
       'teams.home.name', 'teams.home.logo', 'teams.away.id',
       'teams.away.name', 'teams.away.logo', 'scores.home.quarter_1',
       'scores.home.quarter_2', 'scores.home.quarter_3',
       'scores.home.quarter_4', 'scores.home.overtime', 'scores.home.total',
       'scores.away.quarter_1', 'scores.away.quarter_2',
       'scores.away.quarter_3', 'scores.away.quarter_4',
       'scores.away.overtime', 'scores.away.total'],
      dtype='object')

Index(['team.id', 'team.name', 'team.logo', 'statistics.first_downs.total',
       'statistics.first_downs.passing', 'statistics.first_downs.rushing',
       'statistics.first_downs.from_penalties',
       'statistics.first_downs.third_down_efficiency',
       'statistics.first_downs.fourth_down_efficiency',
       'statistics.plays.total', 'statistics.yards.total',
       'statistics.yards.yards_per_play', 'statistics.yards.total_drives',
       'statistics.passing.total', 'statistics.passing.comp_att',
       'statistics.passing.yards_per_pass',
       'statistics.passing.interceptions_thrown',
       'statistics.passing.sacks_yards_lost', 'statistics.rushings.total',
       'statistics.rushings.attempts', 'statistics.rushings.yards_per_rush',
       'statistics.red_zone.made_att', 'statistics.penalties.total',
       'statistics.turnovers.total', 'statistics.turnovers.lost_fumbles',
       'statistics.turnovers.interceptions', 'statistics.posession.total',
       'statistics.interceptions.total', 'statistics.fumbles_recovered.total',
       'statistics.sacks.total', 'statistics.safeties.total',
       'statistics.int_touchdowns.total', 'statistics.points_against.total'],
      dtype='object')

In [ ]:
list1 = ['team.id', 'team.name', 'team.logo', 

'statistics.first_downs.total',
'statistics.first_downs.passing', 
'statistics.first_downs.rushing',
'statistics.first_downs.from_penalties',
'statistics.first_downs.third_down_efficiency',
'statistics.first_downs.fourth_down_efficiency',

'statistics.plays.total', 
'statistics.yards.total',
'statistics.yards.yards_per_play', 
'statistics.yards.total_drives',

'statistics.passing.total', 
'statistics.passing.comp_att', # comp and att
'statistics.passing.yards_per_pass',
'statistics.passing.interceptions_thrown',
'statistics.passing.sacks_yards_lost', #sacks and yards lost

'statistics.rushings.total',
'statistics.rushings.attempts', 
'statistics.rushings.yards_per_rush',

'statistics.red_zone.made_att', 
'statistics.penalties.total',
'statistics.turnovers.total', 
'statistics.turnovers.lost_fumbles',
#'statistics.turnovers.interceptions', 

'statistics.posession.total',
#'statistics.interceptions.total', 
#'statistics.fumbles_recovered.total',

#'statistics.sacks.total', 
'statistics.safeties.total',
'statistics.int_touchdowns.total', 
'statistics.points_against.total'

]
list2 = [
    "date","week","team","opponent","result","Points","points_allowed","overtime",
    "home_game",
    
    "passing_com","passing_att","passing_yds","passing_int", "passing_tds"
    "passing_times_sacked","passing_sack_yards",

    
    "rushing_att","rushing_yds", "rush_tds"
    "fmb",
    "3D_att","3D_conversions",
    "4D_att","4D_conversions", "Time_of_possession",
    
    "XPM","XPA","FGM","FGA",
    #"total_penalties","penalty_yds",
    "punts_total","punts_yds",
    #"punts_blocks",
    "2PM","2PA",
    
    "safety",
    
    "XPR","Pick_6","tds_fmb","tds_KR","tds_PR", "tds_blocked_fg","tds_blocked_punt","tds_walkoff","tds_other", # from events
    
    "1D_passes","1D_runs",
    
    "weekday","season",
    
    #"game_duration_minutes"
]


In [ ]:
# Values used in the rename map
used_values = set(rename_map.values())

# Items in list2 that were NOT used
unused_in_list2 = [item for item in list2 if item not in used_values]

remove_in_games_df = [
    "date",
    "week",
    "Points",
    "points_allowed",
    "home_game",


    "ovetime"]

remove_in_stats = ['opponent']

created_feature = ['result']
    
    
print(unused_in_list2)

['date', 'week', 'opponent', 'result', 'Points', 'overtime', 'home_game', 'passing_com', 'passing_att', 'passing_yds', 'passing_tds', 'passing_sack_yards', 'rushing_att', 'rushing_yds', 'rush_tds', '3D_att', '3D_conversions', '4D_att', '4D_conversions', 'XPM', 'XPA', 'FGM', 'FGA', 'total_penalties', 'penalty_yds', 'punts_total', 'punts_yds', 'punts_blocks', '2PM', '2PA', 'XPR', 'tds_fmb', 'tds_KR', 'tds_PR', 'tds_blocked_fg', 'tds_blocked_punt', 'tds_walkoff', 'tds_other', 'weekday', 'season', 'game_duration_minutes']


In [90]:
# Keys used in the rename map
used_keys = set(rename_map.keys())

# Items in list1 that were NOT used in the rename map
unused_in_list1 = [item for item in list1 if item not in used_keys]

print(unused_in_list1)

['team', 'team', 'team.logo', '1D_passes_runs', '1D_passes', '1D_runs', 'statistics.first_downs.from_penalties', '3D_efficiency', '4D_efficiency', 'statistics.plays.total', 'statistics.turnovers.total', 'fmb', 'passing_int', 'Time_of_possession', 'statistics.interceptions.total', 'statistics.fumbles_recovered.total', 'passing_times_sacked', 'safety', 'Pick_6', 'points_allowed']
